# Mise en place d’une méthode pour la visualisation des représentations internes basées sur des réseaux convolutifs
CNAM certification pro IA - RCP 209 - Semestre 1 2024/2025 - Nicolas Guillard

Project basé sur l'article "Zeiler and al. 2013" © 2024 

<a href="https://creativecommons.org/licenses/by-nc-sa/4.0/?ref=chooser-v1" target="_blank" rel="license noopener noreferrer" style="display:inline-block;">CC BY-NC-SA 4.0<img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/cc.svg?ref=chooser-v1" alt=""><img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/by.svg?ref=chooser-v1" alt=""><img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/nc.svg?ref=chooser-v1" alt=""><img style="height:22px!important;margin-left:3px;vertical-align:text-bottom;" src="https://mirrors.creativecommons.org/presskit/icons/sa.svg?ref=chooser-v1" alt=""></a>

## Modules

In [ ]:
import os
import random
from typing import Tuple
from datetime import datetime

import numpy as np
import torch
import torchvision
from torchvision import transforms as T
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm, trange
from torchinfo import summary
from PIL import Image, ImageDraw

from datasets import imagenet_mean, imagenet_std, DATASET_2, DATASET_1, CustomImageDataset, get_label_data_from_filename
from utils.convnet_wrapper_for_deconvolution import ConvnetWrapperForDeconvolution
from utils.utils_cnn import get_output_sizes, get_receptive_field_in_pixel_space
from utils.utils_images import display_image_tensor as display_image_tensor_, to_0_1, to_0_255,display_images_list_grid, show_display_image_tensor_grid, show_image_tensor
from utils.topk import TopK
from utils.deconvnet import Deconvnet
from utils.utils_rgb import display_rgb_distributions
from imagenet_labels import imagenet1K_labels_to_names


## Variables de contrôle

In [ ]:
# Variables de contrôle d'éxécution du script
test_dataloader = True
test_image_one_by_one = False

# Indexes of layer selected in the paper
# 0 : sortie de la première couche opératoire du modèle
probed_layer_idx = [2, 5, 7, 9, 12] # couche cachées à surveiller
probed_neuron_by_layer = [9, 16, 12, 10, 10] # neurones à surveiller dans chaque couche cachée
K = 9 # Top K

batch_size = 32
SEED = 42 # Random seed for reproducibility

N_images = 2 # Number of images for image by image test
i_stop = 1 # Stop if i_stop > 0

flip_kernels = False
use_bias = False

## Paramétrages globaux

In [ ]:
model_name = "alexnet"
TORCHVISION_MODELS_WEIGHTS = torchvision.models.AlexNet_Weights

Pour la reproductibilité, paramétrage des processus aléatoires.

In [ ]:
random.seed(SEED);
torch.manual_seed(SEED);

## Fonctions propres au carnet

In [ ]:
# Spécifiquement pour un carnet de type Jupyter
def display_image_tensor(
        img_tensor, resize: Tuple[int, int] = None, resample: int = Image.Resampling.NEAREST, verbose=True
        ):
    if display:
        display_image_tensor_(img_tensor, resize=resize, resample=resample, verbose=verbose, fn_display=display)

In [ ]:
def merge_input_and_deconv(background_image, output_deconv, receptive_field, alpha=0.75):
    assert alpha >= 0 and alpha <= 1, "Alpha must be between 0 and 1"

    ((top, left), (bottom, right)) = receptive_field
    background_image = T.functional.to_pil_image(background_image)
    output_deconv = T.functional.to_pil_image(output_deconv).crop((left, top, right, bottom))

    # Ajouter couche alpha pour superposer l'image de sortie sur l'image de fond
    background_image = background_image.convert("RGBA")
    output_resized = output_deconv.convert("RGBA")
    
    # Créer un masque pour la transparence (optionnel)
    mask = Image.new("L", output_resized.size, int(255 * alpha))

    # Coller l'image de sortie sur l'image de fond
    background_image.paste(output_resized, (left, top), mask)

    # Afficher l'image résultante
    #display(background_image)
    return background_image

## Plateforme d'exécution (CPU, GPU (cuda, mps), etc.)

Détection du GPU disponible.

In [ ]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"{device} est disponible")

## Les données

Création du `dataset`et du `dataloader`.

In [ ]:
#DATASET = DATASET_2 # le 1K images pour l'instant
DATASET = DATASET_1 # le 50K images

imagenet_mean = DATASET["means"]
imagenet_std = DATASET["stds"]

geo_transforms = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
])
"""
transforms = T.Compose([
    geo_transforms,
    T.Lambda(lambda t: t/255.), # because read_image -> [0..255]
    T.Normalize(mean=imagenet_mean, std=imagenet_std),
])
"""

transforms = TORCHVISION_MODELS_WEIGHTS.IMAGENET1K_V1.transforms()

get_label_data = lambda f: get_label_data_from_filename(f, DATASET["path"])
dataset_path = DATASET["mounted_path"] if os.path.exists(DATASET["mounted_path"]) else DATASET["path"]
print(f"Utilisation du dataset {DATASET['name']} situé dans {dataset_path}")

dataset = CustomImageDataset(
    dataset_path,
    transform=transforms,
    extension="JPEG",
    dataset_mode=True,
    only_label_idx=False, # On a besoin de l'index pour le nom du fichier
    get_label_data=get_label_data,
    )

dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

En mode test : vérification de la sortie du dataset et du dataloader

In [ ]:
if test_dataloader:
    data = dataset[1]
    print(f"data: {data}")

In [ ]:
if test_dataloader:
    # Dans la configuration (dataset_mode=True, only_label_idx=False) => 
    #   [[image_trfm], [label_idx], [label_code], [idx]]
    i_element = 1
    batch = next(iter(dataloader))

    print("Type batch :", type(batch))
    print("Dimension batch:", len(batch), end="\n\n")
    
    print("Première partie du Batch :", batch[0].size())
    print("Deuxième partie du Batch :", batch[1].size())
    print("Troisième partie du Batch :", len(batch[2]))
    print("Quatrième partie du Batch :", batch[3].size(), end="\n\n")
    
    print("Image (taille du tenseur):", batch[0][i_element].size())
    print("Idx etiquette :", batch[1][i_element].item())
    print("Code etiquette :", batch[2][i_element])
    print("Id file :", batch[3][i_element].item())

## Le modèle convnet

In [ ]:
model_convnet = torch.hub.load('pytorch/vision', model_name, weights=TORCHVISION_MODELS_WEIGHTS.IMAGENET1K_V1)
model_convnet.eval()
model_for_deconv = ConvnetWrapperForDeconvolution(model_convnet, model_convnet.features)

Résumé de la structure du modèle de type ConvNet

In [ ]:
#print("> Résumé de la structure du modèle de type ConvNet")
print(model_convnet)

Tableau des dimensionnalités des couches cachées du modèle

In [ ]:
# print("> Tableau des dimensionnalités des couches cachées du modèle")
batch_input = dataset[0][0].unsqueeze(dim=0)
print(batch_input.size())
model_for_deconv.set_return_switch_indices(False)
summary(model_convnet, input_size=batch_input.size(), mode="eval")

## Récupération des activations des neurones surveillés

Création des structures topK et choix aléatoire des coordonnées de $N$ neurones surveillés par couche cachée indiquée.

In [ ]:
# Récupération des tailles des sorties afin de pouvoir choisir des coordonnées aléatoires
# dans les sorties des couches
input_size=batch_input.size()
model_for_deconv.set_return_switch_indices(False)
output_sizes = get_output_sizes(model_for_deconv.convnet_features, input_size=input_size, last_2d=False)
print("Dimensions des différentes couches cachées")
for layer_idx, output_size in enumerate(output_sizes):
    print(f"\tcouche {layer_idx:2d} : {output_size}")

topk_activations_by_neuron = {}
coord_activations = {}
for layer_idx, n_neurons in zip(probed_layer_idx, probed_neuron_by_layer):
    coord_activations[layer_idx] = []
    i = 0
    # Randomly select N coordinates in the output of the layer : chanel, row, col
    while i < n_neurons:
        # Choisir une coordonnée aléatoire dans la sortie de la couche
        #chn = rng.integers(output_sizes[layer_idx][0])
        #row = rng.integers(output_sizes[layer_idx][1])
        #col = rng.integers(output_sizes[layer_idx][2])
        chn = random.randint(0, output_sizes[layer_idx][0]-1)
        row = random.randint(0, output_sizes[layer_idx][1]-1)
        col = random.randint(0, output_sizes[layer_idx][2]-1)
        # Vérifier que la coordonnée n'est pas déjà choisie (on ne veut pas de doublons)
        if (chn, row, col) not in coord_activations[layer_idx]:
            coord_activations[layer_idx].append((chn, row, col))
            i += 1
    topk_activations_by_neuron[layer_idx] = {coord: TopK(K) for coord in coord_activations[layer_idx]}
print(f"Coordonnées choisies : {coord_activations}")

Générations des activations

In [ ]:
### TEST "image par image###
if test_image_one_by_one:
    dataset_test = CustomImageDataset(
        dataset_path,
        transform=transforms,
        extension="JPEG",
        dataset_mode=False,
        only_label_idx=False, # On a besoin de l'index pour le nom du fichier
        get_label_data=get_label_data,
        )

    model_for_deconv.to(device)
    all_activations = []
    with torch.no_grad():
        for i in trange(0, N_images):
            image, image_trfm, label_idx, label_code, input_file_idx = dataset_test[i]
            batch_input = image_trfm.unsqueeze(dim=0).to(device)
            
            activations = model_for_deconv.get_activations(batch_input, coord_activations, verbose=True)
            all_activations.append((activations, torch.tensor([label_code]), torch.tensor([input_file_idx])))

    print(f"Toutes les activations : {all_activations}")

In [ ]:
if not test_image_one_by_one:
    BATCH_IMAGES_TRFM = 0
    BATCH_LABEL_INDICES = 1
    BATCH_FILE_INDICES = 3

    if i_stop < 0:
        i_stop = min(len(dataloader) + i_stop, 1)
    total = min(i_stop, len(dataloader)) if i_stop else len(dataloader)

    model_for_deconv.to(device)
    all_activations = []
    i_batch = 0
    with torch.no_grad():
        for batch in tqdm(dataloader, total=total, desc="Propagation", unit="batch"):
            if i_stop and i_batch >= i_stop:
                break
            batch_input = batch[BATCH_IMAGES_TRFM].to(device)
            batch_label_idx = batch[BATCH_LABEL_INDICES]
            batch_input_file_idx = batch[BATCH_FILE_INDICES]

            # On fait passer le batch dans le modèle
            activations = model_for_deconv.get_activations(batch_input, coord_activations, verbose=False)
            # et on récupère les activations des couches sélectionnées
            all_activations.append((activations, batch_label_idx, batch_input_file_idx))

            i_batch += 1

Injection dans les topK

In [ ]:
for activations, batch_label_idx, batch_input_file_idx in all_activations:
    for idx_layer, activations_layer in activations.items():
            # On fait le tri pour chaque coordonnée
            for coord, batch_activations_coord in zip(coord_activations[idx_layer], activations_layer):
                topk_activations_by_neuron[idx_layer][coord].append(
                     batch_activations_coord.tolist(),
                     list(zip(batch_input_file_idx.tolist(), batch_label_idx.tolist()))
                )

In [ ]:
all_max_topk = {}
for idx_layer, coords in topk_activations_by_neuron.items():
    max_topk = 0
    max_topk_file_idx = None
    max_topk_label_idx = None
    max_coord = None
    for coord, topk in coords.items():
        print(f"> Layer {idx_layer} neuron {coord} :", topk)
        if topk[0][0] > max_topk:
            max_topk = topk[0][0]
            max_topk_file_idx, max_topk_label_idx = topk[0][1]
            max_coord = coord
    all_max_topk[idx_layer] = (max_topk, max_topk_file_idx, max_coord, max_topk_label_idx)

In [ ]:
for idx_layer, max_topk in all_max_topk.items():
    max_topk, max_topk_file_idx, max_coord, max_label_idx = max_topk
    print(f"Layer {idx_layer} - neuron {max_coord} : {max_topk} - File index : {max_topk_file_idx} - class: {max_label_idx} : {imagenet1K_labels_to_names[max_label_idx]}")

## Déconvolution

In [ ]:
deconvnet = Deconvnet(model_for_deconv, flip_kernels=flip_kernels, use_bias=use_bias)
print(deconvnet)

### Utils

In [ ]:
inv_normalize = T.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.255],
    std=[1/0.229, 1/0.224, 1/0.255]
)
#inv_normalize = UnNormalize(mean=imagenet_mean, std=imagenet_std)

pil_to_tensor = T.PILToTensor()

### Pour une image

Pour la mise au point de la projection de l'image de déconvolution dans l'espace des pixels de l'image d'entrée.

#### Image originale avant redimensionnement

In [ ]:
idx_layer, coords = list(topk_activations_by_neuron.items())[0]
coord, topk = list(coords.items())[0]
idx_map, row, col = coord
max_activation, (file_idx, label_idx) = topk[0]

image, image_trfm = dataset.get_image(file_idx)
display_image_tensor(image, verbose=True)
probed_neuron = (idx_layer, idx_map, row, col)
idx_layer, idx_map, row, col = probed_neuron

Affichage du champ réceptif du neurone surveillé sélectionné, en prenant en référence l'image après transformation.

In [ ]:
batch_input = image_trfm.unsqueeze(dim=0)
output_sizes = get_output_sizes(model_for_deconv.convnet_features, input_size=batch_input.size())
pixel_space_size = batch_input.size(-2), batch_input.size(-1)
receptive_field = get_receptive_field_in_pixel_space(
    pos=(row, col),
    idx_layer=idx_layer,
    cnn_modules=model_for_deconv.convnet_features,
    output_sizes=output_sizes,
    pixel_space_size=pixel_space_size
)

((top, left), (bottom, right)) = receptive_field
print("Champ réceptif :", receptive_field, "taille :", right-left+1, "x", bottom-top+1)

Ne pas oublier de redimensioner l'image (avec `image = geo_transforms(image)`) à l'origine de celle de l'entrée pour rendre cohérente la position du champ réception.

In [ ]:
image_resized = geo_transforms(image)
batch_input = batch_input.squeeze(dim=0)
image_resized_receptive_field = T.functional.to_pil_image(image_resized)
img_draw = ImageDraw.Draw(image_resized_receptive_field)
img_draw.rectangle([(left, top), (right, bottom)], outline="red")
display(image_resized_receptive_field)

#### Déconvolution de l'activation du neurone surveillé sélectionné

In [ ]:
#deconvnet.to(device)
with torch.no_grad():
    deconvnet.wrapped_convnet.to(device)
    deconvnet.to("cpu")
    start_time = datetime.now()
    output_deconv = deconvnet.deconvolution(
        batch_input.to(device),
        idx_layer=idx_layer,
        idx_map=idx_map,
        pos=(row, col),
        clean_feature_map=True,
        return_pos=False,
        verbose=True
    ).detach().cpu()
    end_time = datetime.now()
    print("Durée de la déconv :", end_time-start_time)

    start_time = datetime.now()
    output_deconv = deconvnet.deconvolution(
        batch_input.to(device),
        idx_layer=idx_layer,
        idx_map=idx_map,
        pos=(row, col),
        clean_feature_map=True,
        return_pos=False,
        verbose=True
        ).detach().cpu()
    end_time = datetime.now()
    print("Durée de la déconv :", end_time-start_time)

In [ ]:
display_image_tensor(output_deconv)
show_image_tensor(output_deconv, figsize=(2, 2))

In [ ]:
display_image_tensor(output_deconv.clamp(0, 1))
show_image_tensor(output_deconv.clamp(0, 1), figsize=(2, 2))

In [ ]:
output_deconv_cropped = output_deconv[:, top:bottom, left:right]
display_image_tensor(output_deconv_cropped, resize=(200, 200))
display_image_tensor(output_deconv_cropped.clamp(0, 1), resize=(200, 200))
show_image_tensor(output_deconv_cropped, verbose=True)

In [ ]:
output_deconv_in_pixel_space = torch.nn.functional.relu(output_deconv_cropped)
display_image_tensor(output_deconv_in_pixel_space, resize=(200, 200))
display_image_tensor(output_deconv_in_pixel_space.clamp(0, 1), resize=(200, 200))

In [ ]:
output_deconv_in_pixel_space = to_0_1(output_deconv_cropped)
display_image_tensor(output_deconv_in_pixel_space, resize=(200, 200))

In [ ]:
display_rgb_distributions(output_deconv_in_pixel_space.numpy().transpose(1, 2, 0)*255)

In [ ]:
output_deconv_in_pixel_space = inv_normalize(output_deconv_cropped)
display_image_tensor(output_deconv_in_pixel_space, resize=(200, 200))
display_image_tensor(output_deconv_in_pixel_space.clamp(0, 1), resize=(200, 200))

In [ ]:
#display_image_tensor(to_0_255(unnormalize(output_deconv, mean=imagenet_mean, std=imagenet_std)), verbose=True)
output_deconv_in_pixel_space = to_0_1(inv_normalize(output_deconv_cropped))
display_image_tensor(output_deconv_in_pixel_space, resize=(200, 200))

In [ ]:
display_rgb_distributions(output_deconv_in_pixel_space.numpy().transpose(1, 2, 0)*255)

In [ ]:
#output_deconv_in_pixel_space = unnormalize(output_deconv, mean=imagenet_mean, std=imagenet_std)
output_deconv_in_pixel_space = to_0_1(inv_normalize(output_deconv))

batch_input = batch_input.squeeze(dim=0)
output_deconv_receptive_field = T.functional.to_pil_image(output_deconv_in_pixel_space)
img_draw = ImageDraw.Draw(output_deconv_receptive_field)
img_draw.rectangle([(left, top), (right, bottom)], outline="red")
display(output_deconv_receptive_field)

Générer les instructions python utilisant les librairies `torch` et `torchvision` pour créer une image par superposition d'une image contenue dans le tenseur output, dont les coordonnées des coins supérieurs gauche et inférieur droit sont données par les coordonnées des coins supérieur gauche et inférieur droit du champ réceptif fournies par (left, top), (right, bottom), et d'une image de fond contenue dans le tenseur image.

In [ ]:
input_and_deconv = merge_input_and_deconv(image_resized, output_deconv_in_pixel_space, receptive_field, alpha=0.95)
display(input_and_deconv)

#### Combinaison des images

In [ ]:
from typing import List, Tuple, Callable
import matplotlib
import matplotlib.pyplot as plt

print(matplotlib.__version__)

def display_images_list_grid_LOCAL(
    images: List[torch.Tensor],
    per_rows: int,
    image_titles: List[str] = None,
    title: str = "",
    figsize: Tuple[int, int] = (12, 12)
    ) -> None:
    rows = len(images) // per_rows
    fig, axs = plt.subplots(rows, per_rows, figsize=figsize, layout='constrained')
    for i, (image, ax) in enumerate(zip(images, axs.flat)):
        ax.set_axis_off()
        if image_titles:
            ax.set_title(image_titles[i])
        if image.size(0) == 1:
            ax.imshow(image.numpy().transpose(1, 2, 0), cmap='gray')
        else:
            ax.imshow(image.numpy().transpose(1, 2, 0))
    if title:
        fig.suptitle(title, fontsize=8.0, y=0.85, va="baseline")
    #plt.tight_layout()
    #plt.subplots_adjust(hspace=1)
    plt.axis("off")
    plt.show()

In [ ]:
images = [
    image_resized,
    pil_to_tensor(image_resized_receptive_field),
    pil_to_tensor(output_deconv_receptive_field),
    #pil_to_tensor(T.functional.to_pil_image(image_resized).crop((left, top, right, bottom))),
    image_resized[:, top:bottom+1, left:right+1],
    #pil_to_tensor(T.functional.to_pil_image(output_deconv_in_pixel_space).crop((left, top, right, bottom))),
    output_deconv_in_pixel_space[:, top:bottom+1, left:right+1],
    pil_to_tensor(input_and_deconv)
    ]
display_images_list_grid_LOCAL(images, 6, 
                               #image_titles=[1, 2, 3, 4, 5, 6],
                               title="Image d'entrée, champ réceptif, image de sortie déconv, image de sortie déconv recadrée",
                               figsize=(8, 2))

## Pour l'ensenble des neurones surveillés

In [ ]:
verbose_lvl = 0
with torch.no_grad():
    deconvnet.wrapped_convnet.to(device)
    deconvnet.to("cpu")

    i_result = 0
    for idx_layer, coords in tqdm(
        topk_activations_by_neuron.items(), total=len(topk_activations_by_neuron), desc="Déconvolutions", unit="couche"
        ):
        for coord, topk in coords.items():
            print(f"=== {model_name} - layer {idx_layer} - neuron {coord} - Top {K}===")
            idx_map, row, col = coord
            
            for i_top, (top_value, (file_idx, label_idx)) in enumerate(topk):
                i_result += 1
                class_name = imagenet1K_labels_to_names[label_idx]

                info1 = f"(#{i_result:3d}) [{model_name} - {idx_layer} - {coord}] Top val {i_top+1} = {top_value+1:.3f} avec img n° {file_idx} ({class_name})"
                if verbose_lvl == 1:
                    print(info1, end="|")

                image, image_trfm = dataset.get_image(file_idx)

                # Champ réceptif
                batch_input = image_trfm.unsqueeze(dim=0)
                deconvnet.wrapped_convnet.to("cpu")
                deconvnet.wrapped_convnet.set_return_switch_indices(False)
                output_sizes = get_output_sizes(
                    deconvnet.wrapped_convnet.convnet_features, input_size=batch_input.size()
                    )
                pixel_space_size = batch_input.size(-2), batch_input.size(-1)
                receptive_field = get_receptive_field_in_pixel_space(
                    pos=(row, col),
                    idx_layer=idx_layer,
                    cnn_modules=model_for_deconv.convnet_features,
                    output_sizes=output_sizes,
                    pixel_space_size=pixel_space_size
                )

                # Drawing the receptive field
                ((top, left), (bottom, right)) = receptive_field
                info2 = f"Champ réceptif : {receptive_field} = {right-left+1} x {bottom-top+1}"
                if verbose_lvl == 1:
                    print(info2)
                image_resized = geo_transforms(image)
                batch_input = batch_input.squeeze(dim=0)
                image_resized_receptive_field = T.functional.to_pil_image(image_resized)
                img_draw = ImageDraw.Draw(image_resized_receptive_field)
                img_draw.rectangle([(left, top), (right, bottom)], outline="red")

                # Déconvolution
                deconvnet.wrapped_convnet.to(device)
                output_deconv = deconvnet.deconvolution(
                    batch_input.to(device),
                    idx_layer=idx_layer,
                    idx_map=idx_map,
                    pos=(row, col),
                    clean_feature_map=True,
                    return_pos=False,
                    verbose=verbose_lvl == 2
                    ).detach().cpu()
                
                # Transformation to put deconv output in the pixel space
                #output_deconv_in_pixel_space = unnormalize(output_deconv, mean=imagenet_mean, std=imagenet_std)
                output_deconv_in_pixel_space = to_0_1(inv_normalize(output_deconv))
                
                # Drawing the receptive field in the deconv output
                output_deconv_receptive_field = T.functional.to_pil_image(output_deconv_in_pixel_space)
                img_draw = ImageDraw.Draw(output_deconv_receptive_field)
                img_draw.rectangle([(left, top), (right, bottom)], outline="red")

                # Merging
                input_and_deconv = merge_input_and_deconv(
                    image_resized, output_deconv_in_pixel_space, receptive_field, alpha=0.95
                    )

                # Displaying the combinaison of images
                images = [
                    image_resized,
                    pil_to_tensor(image_resized_receptive_field),
                    pil_to_tensor(output_deconv_receptive_field),
                    #pil_to_tensor(T.functional.to_pil_image(image_resized).crop((left, top, right, bottom))),
                    image_resized[:, top:bottom+1, left:right+1],
                    #pil_to_tensor(T.functional.to_pil_image(output_deconv_in_pixel_space).crop((left, top, right, bottom))),
                    output_deconv_in_pixel_space[:, top:bottom+1, left:right+1],
                    pil_to_tensor(input_and_deconv),
                    ]
                title = info1 + "|" + info2
                _, _ = display_images_list_grid(images, per_rows=6, figsize=(8, 2), title=title, no_plt_show=True);
            show_display_image_tensor_grid()
